<a href="https://colab.research.google.com/github/ViniciusA97/CodeXGLUE/blob/main/Fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch scikit-learn

In [2]:
# Limpar instalações anteriores (se houver)
!rm -rf CodeXGLUE

# Instalar dependências
!pip install -q transformers torch scikit-learn

# Clonar o repositório corrigido (força download limpo)
!git clone --depth 1 https://github.com/ViniciusA97/CodeXGLUE.git
%cd CodeXGLUE/Code-Code/code-refinement

# Verificar se as correções estão presentes
print("\n🔍 Verificando correções:")
!grep -n "from torch.optim import AdamW" code/run.py
!grep -n "getattr(self.config" code/model.py

print("\n✅ Setup completo!")

Cloning into 'CodeXGLUE'...
remote: Enumerating objects: 481, done.
remote: Counting objects: 100% (481/481), done.
remote: Compressing objects: 100% (410/410), done.
remote: Total 481 (delta 70), reused 330 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (481/481), 158.03 MiB | 11.56 MiB/s, done.
Resolving deltas: 100% (70/70), done.
Updating files: 100% (400/400), done.
/content/CodeXGLUE/Code-Code/code-refinement

🔍 Verificando correções:
40:from torch.optim import AdamW
42:        if getattr(self.config, 'torchscript', False):

✅ Setup completo!


In [3]:
!echo "📊 Estatísticas do Dataset SMALL:"
!wc -l data/small/*.buggy data/small/*.fixed

print("\n📝 Exemplo de código com bug:")
!head -1 data/small/train.buggy-fixed.buggy

print("\n✅ Código corrigido:")
!head -1 data/small/train.buggy-fixed.fixed

📊 Estatísticas do Dataset SMALL:
    5835 data/small/test.buggy-fixed.buggy
   46680 data/small/train.buggy-fixed.buggy
    5835 data/small/valid.buggy-fixed.buggy
    5835 data/small/test.buggy-fixed.fixed
   46680 data/small/train.buggy-fixed.fixed
    5835 data/small/valid.buggy-fixed.fixed
  116700 total

📝 Exemplo de código com bug:
public java.lang.String METHOD_1 ( ) { return new TYPE_1 ( STRING_1 ) . format ( VAR_1 [ ( ( VAR_1 . length ) - 1 ) ] . getTime ( ) ) ; } 

✅ Código corrigido:
public java.lang.String METHOD_1 ( ) { return new TYPE_1 ( STRING_1 ) . format ( VAR_1 [ ( ( type ) - 1 ) ] . getTime ( ) ) ; } 


In [4]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU disponível: {torch.cuda.get_device_name(0)}")
    print(f"   Memória total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU não disponível. Configure em Runtime → Change runtime type → GPU")

✅ GPU disponível: Tesla T4
   Memória total: 15.64 GB


In [8]:
%cd /content/CodeXGLUE/Code-Code/code-refinement/code

!python run.py \
    --do_train \
    --do_eval \
    --model_type roberta \
    --model_name_or_path microsoft/codebert-base \
    --config_name roberta-base \
    --tokenizer_name roberta-base \
    --train_filename ../data/small/train.buggy-fixed.buggy,../data/small/train.buggy-fixed.fixed \
    --dev_filename ../data/small/valid.buggy-fixed.buggy,../data/small/valid.buggy-fixed.fixed \
    --output_dir ./output_small \
    --max_source_length 128 \
    --max_target_length 128 \
    --beam_size 3 \
    --train_batch_size 48 \
    --eval_batch_size 48 \
    --learning_rate 5e-5 \
    --train_steps 10000 \
    --eval_steps 5000

print("\n✅ Treinamento concluído!")

/content/CodeXGLUE/Code-Code/code-refinement/code
03/25/2026 09:14:25 - INFO - __main__ -   Namespace(model_type='roberta', model_name_or_path='microsoft/codebert-base', tokenizer_name='roberta-base', output_dir='./output_small', load_model_path=None, train_filename='../data/small/train.buggy-fixed.buggy,../data/small/train.buggy-fixed.fixed', dev_filename='../data/small/valid.buggy-fixed.buggy,../data/small/valid.buggy-fixed.fixed', test_filename=None, config_name='roberta-base', max_source_length=128, max_target_length=128, do_train=True, do_eval=True, do_test=False, do_lower_case=False, no_cuda=False, train_batch_size=48, eval_batch_size=48, gradient_accumulation_steps=1, learning_rate=5e-05, beam_size=3, weight_decay=0.0, adam_epsilon=1e-08, max_grad_norm=1.0, num_train_epochs=3.0, max_steps=-1, eval_steps=5000, train_steps=10000, warmup_steps=0, local_rank=-1, seed=42)
03/25/2026 09:14:25 - WARNING - __main__ -   Process rank: -1, device: cuda, n_gpu: 1, distributed training: Fals

In [2]:
%cd /content/CodeXGLUE/Code-Code/code-refinement/code
!python run.py \
    --do_test \
    --model_type roberta \
    --model_name_or_path roberta-base \
    --config_name roberta-base \
    --tokenizer_name roberta-base \
    --load_model_path ./output_small/checkpoint-best-bleu/pytorch_model.bin \
    --dev_filename ../data/small/valid.buggy-fixed.buggy,../data/small/valid.buggy-fixed.fixed \
    --test_filename ../data/small/test.buggy-fixed.buggy,../data/small/test.buggy-fixed.fixed \
    --output_dir ./output_small \
    --max_source_length 256 \
    --max_target_length 256 \
    --beam_size 5 \
    --eval_batch_size 16

print("\n✅ Inferência concluída!")

[Errno 2] No such file or directory: '/content/CodeXGLUE/Code-Code/code-refinement/code'
/content
python3: can't open file '/content/run.py': [Errno 2] No such file or directory

✅ Inferência concluída!


In [ ]:
%cd ..

!python evaluator/evaluator.py \
    -ref data/small/test.buggy-fixed.fixed \
    -pre code/output_small/test_0.output

print("\n📈 Resultados esperados: BLEU: ~77.42, Acc: ~16.4%")

In [ ]:
print("🐛 CÓDIGO COM BUG (input):")
!head -5 data/small/test.buggy-fixed.buggy

print("\n✅ CÓDIGO CORRIGIDO REAL (ground truth):")
!head -5 data/small/test.buggy-fixed.fixed

print("\n🤖 PREDIÇÃO DO MODELO:")
!head -5 code/output_small/test_0.output

In [3]:
# Compactar o modelo treinado
!zip -r model_trained.zip code/output_small/checkpoint-best-bleu/

# Download (aparecerá no painel esquerdo do Colab)
from google.colab import files
files.download('model_trained.zip')

print("✅ Modelo compactado! Faça o download do arquivo 'model_trained.zip'")

	zip warning: name not matched: code/output_small/checkpoint-best-bleu/

zip error: Nothing to do! (try: zip -r model_trained.zip . -i code/output_small/checkpoint-best-bleu/)


FileNotFoundError: Cannot find file: model_trained.zip